# 02 — JMG Coordinate Resolution

**FYP:** Developing a Semi-Dynamic Landslide Susceptibility Decision-Support Dashboard Using Machine Learning for Malaysia

## Purpose

Stage 1 showed that most `WGSe/WGSn` values in the 52-row JMG CSV are too coarse for point-level modelling.  
The original JMG landslide shapefiles are now available and contain precise projected geometry.

This notebook therefore:

1. Loads the 52-row JMG CSV.
2. Locates the **five actual JMG landslide occurrence layers**.
3. Reads the source CRS directly from each shapefile.
4. Checks that the five layers total 52 occurrence features.
5. Verifies that CSV `RSOx/RSOy` pairs exist in the corresponding source shapefile.
6. Converts original JMG source geometry to WGS84 (`EPSG:4326`).
7. Validates transformed locations against the Malaysia boundary.
8. Compares transformed coordinates with the original coarse `WGSe/WGSn`.
9. Creates a clean spatial JMG inventory for the next deduplication stage.
10. Saves audit tables and a map.

## Rules

- No CRS is invented.
- The CRS declared by each original JMG shapefile is used.
- `WGSe/WGSn` values are **not** used to reconstruct geometry.
- Original `RSOx/RSOy` values and source attributes are preserved.
- No JMG record is deleted in this notebook.
- Duplicate-event decisions are postponed to Stage 3.
- A repeated projected coordinate is flagged for later review rather than automatically treated as one event.


In [3]:
# OPTIONAL: Only run if packages are missing.
# !pip install -q geopandas pyproj shapely matplotlib pandas numpy

from pathlib import Path
import os
import math
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=UserWarning)

print("pandas:", pd.__version__)
print("geopandas:", gpd.__version__)


pandas: 3.0.2
geopandas: 1.1.4


## 1. Project paths

Change **only `PROJECT_ROOT`**.

Recommended structure:

```text
Landslide_FYP/
├── Landslide_FYP_Data/
│   ├── 01_Inventory/
│   │   └── JMG/
│   │       ├── JMG_Government_Raw_Landslide_Inventory.csv
│   │       └── Original_JMG_Spatial/
│   │           └── Bencana_Geologi/
│   │               ├── uk_TANAH_RUNTUH_SEMENANJUNG_.shp
│   │               ├── ch_TANAH_RUNTUH_SEMENANJUNG_.shp
│   │               ├── pp_TANAH_RUNTUH_SEMENANJUNG_.shp
│   │               ├── ipoh_TANAH_RUNTUH_SEMENANJUNG_.shp
│   │               └── kdsg_TANAH_RUNTUH_SABAH.shp
│   └── 02_Boundary/
│       └── mys_admin0.geojson
│
└── Landslide_FYP_Development/
    ├── 01_preprocessing/
    └── 02_cleaned_inventory/
```

If your extracted folder is named `Bencana Geologi` instead of `Bencana_Geologi`, the notebook will automatically search for it.


In [4]:
# ============================================================
# CHANGE ONLY THIS LINE
# ============================================================
PROJECT_ROOT = Path(r"C:\Landslide_FYP")

RAW_DIR = PROJECT_ROOT / "Landslide_FYP_Data"
DEV_DIR = PROJECT_ROOT / "Landslide_FYP_Development"

JMG_CSV_PATH = (
    RAW_DIR
    / "01_Inventory"
    / "JMG"
    / "JMG_Government_Raw_Landslide_Inventory.csv"
)

BOUNDARY_PATH = (
    RAW_DIR
    / "02_Boundary"
    / "mys_admin0.geojson"
)

JMG_RAW_FOLDER = (
    RAW_DIR
    / "01_Inventory"
    / "JMG"
    / "Original_JMG_Spatial"
)

OUTPUT_DIR = (
    DEV_DIR
    / "01_preprocessing"
    / "jmg_coordinate_resolution_outputs"
)

CLEAN_DIR = DEV_DIR / "02_cleaned_inventory"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

print("JMG CSV:", JMG_CSV_PATH)
print("JMG spatial root:", JMG_RAW_FOLDER)
print("Malaysia boundary:", BOUNDARY_PATH)
print("Audit output:", OUTPUT_DIR)
print("Cleaned inventory:", CLEAN_DIR)

if not JMG_CSV_PATH.exists():
    raise FileNotFoundError(
        f"JMG CSV not found:\\n{JMG_CSV_PATH}"
    )

if not JMG_RAW_FOLDER.exists():
    raise FileNotFoundError(
        f"JMG spatial folder not found:\\n{JMG_RAW_FOLDER}\\n\\n"
        "Extract Bencana Geologi.zip into this folder first."
    )

if not BOUNDARY_PATH.exists():
    raise FileNotFoundError(
        f"Malaysia boundary not found:\\n{BOUNDARY_PATH}"
    )


JMG CSV: C:\Landslide_FYP\Landslide_FYP_Data\01_Inventory\JMG\JMG_Government_Raw_Landslide_Inventory.csv
JMG spatial root: C:\Landslide_FYP\Landslide_FYP_Data\01_Inventory\JMG\Original_JMG_Spatial
Malaysia boundary: C:\Landslide_FYP\Landslide_FYP_Data\02_Boundary\mys_admin0.geojson
Audit output: C:\Landslide_FYP\Landslide_FYP_Development\01_preprocessing\jmg_coordinate_resolution_outputs
Cleaned inventory: C:\Landslide_FYP\Landslide_FYP_Development\02_cleaned_inventory


## 2. Define the five occurrence layers

These are the five layers that correspond to the 52-row JMG occurrence CSV.

Other layers in `Bencana Geologi` such as hotspot, debris flow, engineering geology, slope stability, earthquake, subsidence and sinkhole are **not loaded as occurrence records**.


In [5]:
EXPECTED_LAYERS = {
    "uk_TANAH_RUNTUH_SEMENANJUNG_.shp": 28,
    "ch_TANAH_RUNTUH_SEMENANJUNG_.shp": 16,
    "pp_TANAH_RUNTUH_SEMENANJUNG_.shp": 3,
    "ipoh_TANAH_RUNTUH_SEMENANJUNG_.shp": 1,
    "kdsg_TANAH_RUNTUH_SABAH.shp": 4,
}

EXPECTED_TOTAL = sum(EXPECTED_LAYERS.values())

print("Expected source layers:", len(EXPECTED_LAYERS))
print("Expected total occurrence records:", EXPECTED_TOTAL)


Expected source layers: 5
Expected total occurrence records: 52


## 3. Automatically locate the five shapefiles

The code searches recursively under `Original_JMG_Spatial`, so the extracted folder may be named either `Bencana Geologi` or `Bencana_Geologi`.


In [6]:
def find_unique_file(root, filename):
    matches = list(root.rglob(filename))
    if len(matches) == 0:
        raise FileNotFoundError(
            f"Could not find required file: {filename}\n"
            f"Searched under: {root}"
        )
    if len(matches) > 1:
        print(f"WARNING: Multiple matches for {filename}:")
        for m in matches:
            print(" -", m)
        raise RuntimeError(
            "Keep only one copy of each original occurrence shapefile "
            "under the JMG raw spatial folder."
        )
    return matches[0]

layer_paths = {
    filename: find_unique_file(JMG_RAW_FOLDER, filename)
    for filename in EXPECTED_LAYERS
}

for filename, path in layer_paths.items():
    print(filename)
    print(" ->", path)


uk_TANAH_RUNTUH_SEMENANJUNG_.shp
 -> C:\Landslide_FYP\Landslide_FYP_Data\01_Inventory\JMG\Original_JMG_Spatial\Bencana Geologi\uk_TANAH_RUNTUH_SEMENANJUNG_.shp
ch_TANAH_RUNTUH_SEMENANJUNG_.shp
 -> C:\Landslide_FYP\Landslide_FYP_Data\01_Inventory\JMG\Original_JMG_Spatial\Bencana Geologi\ch_TANAH_RUNTUH_SEMENANJUNG_.shp
pp_TANAH_RUNTUH_SEMENANJUNG_.shp
 -> C:\Landslide_FYP\Landslide_FYP_Data\01_Inventory\JMG\Original_JMG_Spatial\Bencana Geologi\pp_TANAH_RUNTUH_SEMENANJUNG_.shp
ipoh_TANAH_RUNTUH_SEMENANJUNG_.shp
 -> C:\Landslide_FYP\Landslide_FYP_Data\01_Inventory\JMG\Original_JMG_Spatial\Bencana Geologi\ipoh_TANAH_RUNTUH_SEMENANJUNG_.shp
kdsg_TANAH_RUNTUH_SABAH.shp
 -> C:\Landslide_FYP\Landslide_FYP_Data\01_Inventory\JMG\Original_JMG_Spatial\Bencana Geologi\kdsg_TANAH_RUNTUH_SABAH.shp


## 4. Load JMG CSV and Malaysia boundary

In [7]:
jmg_csv = pd.read_csv(JMG_CSV_PATH)
malaysia = gpd.read_file(BOUNDARY_PATH)

print("JMG CSV shape:", jmg_csv.shape)
print("Malaysia boundary shape:", malaysia.shape)
print("Malaysia CRS:", malaysia.crs)

if len(jmg_csv) != EXPECTED_TOTAL:
    raise ValueError(
        f"Expected {EXPECTED_TOTAL} JMG CSV rows, but found {len(jmg_csv)}."
    )

required_csv_cols = [
    "SOURCE_FILE",
    "RSOx",
    "RSOy",
    "WGSe",
    "WGSn",
]

missing = [c for c in required_csv_cols if c not in jmg_csv.columns]
if missing:
    raise KeyError(f"Required JMG CSV columns missing: {missing}")

print("\nCSV records by SOURCE_FILE:")
display(
    jmg_csv["SOURCE_FILE"]
    .value_counts(dropna=False)
    .rename_axis("SOURCE_FILE")
    .reset_index(name="csv_count")
)


JMG CSV shape: (52, 41)
Malaysia boundary shape: (1, 18)
Malaysia CRS: EPSG:4326

CSV records by SOURCE_FILE:


,SOURCE_FILE,csv_count
0,uk_TANAH_RUNTUH_SEMENANJUNG_.shp,28
1,ch_TANAH_RUNTUH_SEMENANJUNG_.shp,16
2,kdsg_TANAH_RUNTUH_SABAH.shp,4
3,pp_TANAH_RUNTUH_SEMENANJUNG_.shp,3
4,ipoh_TANAH_RUNTUH_SEMENANJUNG_.shp,1


## 5. Inspect source layer CRS and feature counts

This is the critical safety check.  
The notebook does not assign a CRS manually; it reads the CRS declared by each source shapefile.


In [8]:
source_layers = {}
crs_rows = []

for filename, path in layer_paths.items():
    gdf = gpd.read_file(path)

    if gdf.crs is None:
        raise ValueError(
            f"CRS is missing from source layer: {filename}. "
            "Stop here rather than guessing."
        )

    if not all(gdf.geometry.geom_type.isin(["Point", "Point Z"])):
        # GeoPandas normally reports Point even when Z exists.
        if not all(gdf.geometry.geom_type == "Point"):
            raise ValueError(
                f"{filename} contains unexpected geometry types: "
                f"{gdf.geometry.geom_type.value_counts().to_dict()}"
            )

    source_layers[filename] = gdf

    crs_rows.append({
        "SOURCE_FILE": filename,
        "expected_features": EXPECTED_LAYERS[filename],
        "actual_features": len(gdf),
        "crs": str(gdf.crs),
        "epsg": gdf.crs.to_epsg(),
        "geometry_type": ", ".join(sorted(gdf.geom_type.unique())),
        "min_x": gdf.total_bounds[0],
        "min_y": gdf.total_bounds[1],
        "max_x": gdf.total_bounds[2],
        "max_y": gdf.total_bounds[3],
    })

crs_inventory = pd.DataFrame(crs_rows)

display(crs_inventory)

crs_inventory.to_csv(
    OUTPUT_DIR / "jmg_source_crs_inventory.csv",
    index=False
)

if not (crs_inventory["expected_features"] == crs_inventory["actual_features"]).all():
    raise ValueError(
        "At least one JMG source layer does not contain the expected "
        "number of occurrence features. Review the files before continuing."
    )

if crs_inventory["actual_features"].sum() != EXPECTED_TOTAL:
    raise ValueError(
        f"Source layers total {crs_inventory['actual_features'].sum()} records, "
        f"expected {EXPECTED_TOTAL}."
    )

print("\nAll five source feature counts match the 52-row inventory.")


,SOURCE_FILE,expected_features,actual_features,crs,epsg,geometry_type,min_x,min_y,max_x,max_y
0,uk_TANAH_RUNTUH_SEMENANJUNG_.shp,28,28,EPSG:3375,3375,Point,414215.01,343063.96,428333.43,359233.2
1,ch_TANAH_RUNTUH_SEMENANJUNG_.shp,16,16,EPSG:3375,3375,Point,375495.00,487926.00,378718.00,497983.0
2,pp_TANAH_RUNTUH_SEMENANJUNG_.shp,3,3,EPSG:3375,3375,Point,254568.00,603963.50,256719.82,605378.0
3,ipoh_TANAH_RUNTUH_SEMENANJUNG_.shp,1,1,EPSG:3375,3375,Point,341640.00,514191.00,341640.00,514191.0
4,kdsg_TANAH_RUNTUH_SABAH.shp,4,4,EPSG:3376,3376,Point,760166.00,664729.00,766754.00,666458.0



All five source feature counts match the 52-row inventory.


## 6. Verify expected declared CRS

From the supplied JMG source files, the expected pattern is:

- Peninsular occurrence layers → `EPSG:3375`
- Sabah occurrence layer → `EPSG:3376`

This cell checks that pattern.  
It still uses the CRS declared by the source files, not a manually assigned CRS.


In [9]:
expected_epsg = {
    "uk_TANAH_RUNTUH_SEMENANJUNG_.shp": 3375,
    "ch_TANAH_RUNTUH_SEMENANJUNG_.shp": 3375,
    "pp_TANAH_RUNTUH_SEMENANJUNG_.shp": 3375,
    "ipoh_TANAH_RUNTUH_SEMENANJUNG_.shp": 3375,
    "kdsg_TANAH_RUNTUH_SABAH.shp": 3376,
}

crs_inventory["expected_epsg"] = crs_inventory["SOURCE_FILE"].map(expected_epsg)
crs_inventory["epsg_matches_expected"] = (
    crs_inventory["epsg"] == crs_inventory["expected_epsg"]
)

display(
    crs_inventory[
        ["SOURCE_FILE", "actual_features", "epsg",
         "expected_epsg", "epsg_matches_expected"]
    ]
)

if not crs_inventory["epsg_matches_expected"].all():
    raise ValueError(
        "One or more source CRS values differ from the CRS previously "
        "observed in the supplied JMG files. Stop and review."
    )

print("CRS checks passed.")


,SOURCE_FILE,actual_features,epsg,expected_epsg,epsg_matches_expected
0,uk_TANAH_RUNTUH_SEMENANJUNG_.shp,28,3375,3375,True
1,ch_TANAH_RUNTUH_SEMENANJUNG_.shp,16,3375,3375,True
2,pp_TANAH_RUNTUH_SEMENANJUNG_.shp,3,3375,3375,True
3,ipoh_TANAH_RUNTUH_SEMENANJUNG_.shp,1,3375,3375,True
4,kdsg_TANAH_RUNTUH_SABAH.shp,4,3376,3376,True


CRS checks passed.


## 7. Check source geometry against source `RSOx/RSOy`

The projected point geometry should agree with the source attribute coordinates.  
Small floating-point differences are acceptable; large differences are not.


In [10]:
geometry_coordinate_checks = []

for filename, gdf in source_layers.items():
    if "RSOx" not in gdf.columns or "RSOy" not in gdf.columns:
        raise KeyError(
            f"{filename} does not contain RSOx/RSOy fields."
        )

    temp = gdf.copy()
    temp["RSOx_num"] = pd.to_numeric(temp["RSOx"], errors="coerce")
    temp["RSOy_num"] = pd.to_numeric(temp["RSOy"], errors="coerce")

    temp["geometry_x"] = temp.geometry.x
    temp["geometry_y"] = temp.geometry.y

    temp["dx_m"] = (temp["geometry_x"] - temp["RSOx_num"]).abs()
    temp["dy_m"] = (temp["geometry_y"] - temp["RSOy_num"]).abs()
    temp["xy_difference_m"] = np.sqrt(
        temp["dx_m"] ** 2 + temp["dy_m"] ** 2
    )

    for source_index, row in temp.iterrows():
        geometry_coordinate_checks.append({
            "SOURCE_FILE": filename,
            "source_feature_index": source_index,
            "RSOx": row["RSOx_num"],
            "RSOy": row["RSOy_num"],
            "geometry_x": row["geometry_x"],
            "geometry_y": row["geometry_y"],
            "xy_difference_m": row["xy_difference_m"],
        })

geometry_check = pd.DataFrame(geometry_coordinate_checks)

display(
    geometry_check.sort_values(
        "xy_difference_m",
        ascending=False
    ).head(20)
)

geometry_check.to_csv(
    OUTPUT_DIR / "jmg_geometry_vs_rso_check.csv",
    index=False
)

print(
    "Maximum geometry-vs-RSO difference:",
    geometry_check["xy_difference_m"].max(),
    "projected units"
)

if geometry_check["xy_difference_m"].max() > 1.0:
    print(
        "WARNING: At least one geometry differs from RSOx/RSOy by > 1 unit."
    )
else:
    print("Source point geometry agrees with RSOx/RSOy.")


,SOURCE_FILE,source_feature_index,RSOx,RSOy,geometry_x,geometry_y,xy_difference_m
0,uk_TANAH_RUNTUH_SEMENANJUNG_.shp,0,416717.63,343063.96,416717.63,343063.96,0.0
1,uk_TANAH_RUNTUH_SEMENANJUNG_.shp,1,417244.68,344799.69,417244.68,344799.69,0.0
2,uk_TANAH_RUNTUH_SEMENANJUNG_.shp,2,414215.01,348569.97,414215.01,348569.97,0.0
3,uk_TANAH_RUNTUH_SEMENANJUNG_.shp,3,420619.31,348824.82,420619.31,348824.82,0.0
4,uk_TANAH_RUNTUH_SEMENANJUNG_.shp,4,420128.42,347480.63,420128.42,347480.63,0.0
5,uk_TANAH_RUNTUH_SEMENANJUNG_.shp,5,419960.26,345501.44,419960.26,345501.44,0.0
6,uk_TANAH_RUNTUH_SEMENANJUNG_.shp,6,420680.48,348573.13,420680.48,348573.13,0.0
7,uk_TANAH_RUNTUH_SEMENANJUNG_.shp,7,421874.77,348328.99,421874.77,348328.99,0.0
8,uk_TANAH_RUNTUH_SEMENANJUNG_.shp,8,424567.14,345624.94,424567.14,345624.94,0.0
9,uk_TANAH_RUNTUH_SEMENANJUNG_.shp,9,428333.43,344009.91,428333.43,344009.91,0.0


Maximum geometry-vs-RSO difference: 0.0 projected units
Source point geometry agrees with RSOx/RSOy.


## 8. Cross-check CSV rows against source shapefiles

A CSV row is matched using:

1. `SOURCE_FILE`
2. exact/near-exact `RSOx`
3. exact/near-exact `RSOy`

Tolerance = **0.05 projected units**.

This check is designed to confirm that the CSV came from these five source layers.  
It does not yet decide whether two records at the same point are duplicate events.


In [11]:
MATCH_TOLERANCE = 0.05

jmg_csv_work = jmg_csv.copy()
jmg_csv_work.insert(
    0,
    "csv_audit_id",
    [f"JMG_CSV_{i:03d}" for i in range(1, len(jmg_csv_work) + 1)]
)

jmg_csv_work["RSOx"] = pd.to_numeric(
    jmg_csv_work["RSOx"], errors="coerce"
)
jmg_csv_work["RSOy"] = pd.to_numeric(
    jmg_csv_work["RSOy"], errors="coerce"
)

csv_match_rows = []

for _, csv_row in jmg_csv_work.iterrows():
    filename = csv_row["SOURCE_FILE"]

    if filename not in source_layers:
        csv_match_rows.append({
            "csv_audit_id": csv_row["csv_audit_id"],
            "SOURCE_FILE": filename,
            "csv_RSOx": csv_row["RSOx"],
            "csv_RSOy": csv_row["RSOy"],
            "candidate_source_matches": 0,
            "nearest_distance": np.nan,
            "match_status": "SOURCE_FILE_NOT_LOADED",
        })
        continue

    src = source_layers[filename].copy()

    sx = pd.to_numeric(src["RSOx"], errors="coerce")
    sy = pd.to_numeric(src["RSOy"], errors="coerce")

    dist = np.sqrt(
        (sx - csv_row["RSOx"]) ** 2
        + (sy - csv_row["RSOy"]) ** 2
    )

    candidate_mask = dist <= MATCH_TOLERANCE
    candidate_count = int(candidate_mask.sum())
    nearest_distance = float(dist.min()) if len(dist) else np.nan

    if candidate_count == 1:
        status = "UNIQUE_RSO_MATCH"
    elif candidate_count > 1:
        status = "AMBIGUOUS_REPEATED_RSO"
    else:
        status = "NO_RSO_MATCH"

    csv_match_rows.append({
        "csv_audit_id": csv_row["csv_audit_id"],
        "SOURCE_FILE": filename,
        "csv_RSOx": csv_row["RSOx"],
        "csv_RSOy": csv_row["RSOy"],
        "candidate_source_matches": candidate_count,
        "nearest_distance": nearest_distance,
        "match_status": status,
    })

csv_source_match = pd.DataFrame(csv_match_rows)

display(
    csv_source_match["match_status"]
    .value_counts(dropna=False)
    .rename_axis("match_status")
    .reset_index(name="count")
)

display(
    csv_source_match[
        csv_source_match["match_status"] != "UNIQUE_RSO_MATCH"
    ]
)

csv_source_match.to_csv(
    OUTPUT_DIR / "jmg_csv_to_source_match_audit.csv",
    index=False
)

if (csv_source_match["match_status"] == "NO_RSO_MATCH").any():
    raise ValueError(
        "At least one CSV RSO coordinate could not be found in its "
        "declared source layer. Review the audit table."
    )

print(
    "\nAll CSV coordinate pairs exist in their declared source layer. "
    "Repeated RSO locations remain flagged for Stage 3 review."
)


,match_status,count
0,UNIQUE_RSO_MATCH,50
1,AMBIGUOUS_REPEATED_RSO,2


,csv_audit_id,SOURCE_FILE,csv_RSOx,csv_RSOy,candidate_source_matches,nearest_distance,match_status
41,JMG_CSV_042,ch_TANAH_RUNTUH_SEMENANJUNG_.shp,377057.0,497079.0,2,0.0,AMBIGUOUS_REPEATED_RSO
42,JMG_CSV_043,ch_TANAH_RUNTUH_SEMENANJUNG_.shp,377057.0,497079.0,2,0.0,AMBIGUOUS_REPEATED_RSO



All CSV coordinate pairs exist in their declared source layer. Repeated RSO locations remain flagged for Stage 3 review.


## 9. Build one authoritative 52-point source inventory

For spatial geometry, the **original source shapefile geometry** is treated as authoritative.

The five layers are converted separately to WGS84 and then concatenated.


In [12]:
resolved_parts = []

for filename, gdf in source_layers.items():
    original = gdf.copy()

    # Preserve source feature index before reprojection/concatenation.
    original["source_feature_index"] = original.index.astype(int)
    original["SOURCE_FILE"] = filename
    original["source_crs"] = str(gdf.crs)
    original["source_epsg"] = gdf.crs.to_epsg()

    # Preserve exact original geometry coordinate values.
    original["source_geometry_x"] = original.geometry.x
    original["source_geometry_y"] = original.geometry.y

    # Reproject using source-declared CRS.
    wgs = original.to_crs("EPSG:4326")

    wgs["resolved_longitude"] = wgs.geometry.x
    wgs["resolved_latitude"] = wgs.geometry.y

    resolved_parts.append(wgs)

resolved = gpd.GeoDataFrame(
    pd.concat(resolved_parts, ignore_index=True),
    geometry="geometry",
    crs="EPSG:4326"
)

resolved.insert(
    0,
    "jmg_resolved_id",
    [f"JMG_{i:03d}" for i in range(1, len(resolved) + 1)]
)

print("Resolved feature count:", len(resolved))
print("Resolved CRS:", resolved.crs)
print(
    "Geometry types:",
    resolved.geom_type.value_counts().to_dict()
)

if len(resolved) != EXPECTED_TOTAL:
    raise ValueError(
        f"Expected {EXPECTED_TOTAL} resolved points, found {len(resolved)}."
    )


Resolved feature count: 52
Resolved CRS: EPSG:4326
Geometry types: {'Point': 52}


## 10. Create safe standard ID field

Peninsular files use `ID_Tanah_R`; the Sabah file uses `ID__`.

Both are retained in the raw source attributes.  
This cell creates one standard `jmg_source_event_id` without inventing missing IDs.


In [13]:
resolved["jmg_source_event_id"] = pd.NA

if "ID_Tanah_R" in resolved.columns:
    resolved["jmg_source_event_id"] = (
        resolved["jmg_source_event_id"]
        .fillna(resolved["ID_Tanah_R"])
    )

if "ID__" in resolved.columns:
    resolved["jmg_source_event_id"] = (
        resolved["jmg_source_event_id"]
        .fillna(resolved["ID__"])
    )

resolved["jmg_source_event_id"] = (
    resolved["jmg_source_event_id"]
    .astype("string")
    .str.strip()
    .replace({"": pd.NA, "None": pd.NA, "nan": pd.NA})
)

print(
    "Non-missing JMG source event IDs:",
    resolved["jmg_source_event_id"].notna().sum(),
    "/",
    len(resolved)
)

print(
    "Missing JMG source event IDs:",
    resolved["jmg_source_event_id"].isna().sum()
)


Non-missing JMG source event IDs: 49 / 52
Missing JMG source event IDs: 3


## 11. Preserve source date fields and create a date candidate

This is **not yet a rainfall eligibility flag**.

A date candidate is created only if the source provides:
- valid day,
- recognised month,
- valid year.

No missing day/month is filled.


In [14]:
month_map = {
    "januari": 1, "january": 1,
    "februari": 2, "february": 2,
    "mac": 3, "march": 3,
    "april": 4,
    "mei": 5, "may": 5,
    "jun": 6, "june": 6,
    "julai": 7, "july": 7,
    "ogos": 8, "august": 8,
    "september": 9,
    "oktober": 10, "october": 10,
    "november": 11,
    "disember": 12, "december": 12,
}

resolved["_day"] = pd.to_numeric(
    resolved.get("Hari"), errors="coerce"
)

resolved["_year"] = pd.to_numeric(
    resolved.get("Tahun"), errors="coerce"
)

resolved["_month"] = (
    resolved.get("Bulan")
    .astype("string")
    .str.strip()
    .str.lower()
    .map(month_map)
)

complete_parts = (
    resolved["_day"].between(1, 31)
    & resolved["_month"].between(1, 12)
    & resolved["_year"].between(1900, 2100)
)

resolved["source_date_candidate"] = pd.NaT

date_values = pd.to_datetime(
    pd.DataFrame({
        "year": resolved.loc[complete_parts, "_year"],
        "month": resolved.loc[complete_parts, "_month"],
        "day": resolved.loc[complete_parts, "_day"],
    }),
    errors="coerce"
)

resolved.loc[
    complete_parts,
    "source_date_candidate"
] = date_values.values

resolved["source_date_complete_valid"] = (
    resolved["source_date_candidate"].notna()
)

print(
    "Complete valid source dates:",
    resolved["source_date_complete_valid"].sum(),
    "/",
    len(resolved)
)

print(
    "Incomplete / invalid source dates:",
    (~resolved["source_date_complete_valid"]).sum()
)


Complete valid source dates: 47 / 52
Incomplete / invalid source dates: 5


## 12. Compare original JMG `WGSe/WGSn` with resolved WGS84

The original `WGSe/WGSn` fields are preserved for provenance.

Because many source values are whole degrees or zero placeholders, they are **not** used as model coordinates.

This section only measures whether the source WGS fields are:
- zero/missing placeholder,
- whole-degree coarse values,
- decimal coordinates.


In [15]:
resolved["original_WGSe_numeric"] = pd.to_numeric(
    resolved.get("WGSe"), errors="coerce"
)
resolved["original_WGSn_numeric"] = pd.to_numeric(
    resolved.get("WGSn"), errors="coerce"
)

def classify_original_wgs(row):
    lon = row["original_WGSe_numeric"]
    lat = row["original_WGSn_numeric"]

    if pd.isna(lon) or pd.isna(lat):
        return "MISSING"
    if lon == 0 and lat == 0:
        return "ZERO_PLACEHOLDER"

    fractional = (
        abs(lon % 1) > 1e-6
        or abs(lat % 1) > 1e-6
    )

    if fractional:
        return "DECIMAL_COORDINATE"

    return "WHOLE_DEGREE_COARSE"

resolved["original_wgs_status"] = resolved.apply(
    classify_original_wgs,
    axis=1
)

# Difference is informative only for decimal source WGS values.
resolved["original_vs_resolved_distance_m"] = np.nan

decimal_mask = (
    resolved["original_wgs_status"] == "DECIMAL_COORDINATE"
)

if decimal_mask.any():
    source_wgs_points = gpd.GeoSeries(
        gpd.points_from_xy(
            resolved.loc[decimal_mask, "original_WGSe_numeric"],
            resolved.loc[decimal_mask, "original_WGSn_numeric"],
        ),
        crs="EPSG:4326"
    ).to_crs("EPSG:3857")

    resolved_points = gpd.GeoSeries(
        resolved.loc[decimal_mask, "geometry"],
        crs="EPSG:4326"
    ).to_crs("EPSG:3857")

    resolved.loc[
        decimal_mask,
        "original_vs_resolved_distance_m"
    ] = source_wgs_points.distance(
        resolved_points,
        align=False
    ).to_numpy()

display(
    resolved["original_wgs_status"]
    .value_counts(dropna=False)
    .rename_axis("original_wgs_status")
    .reset_index(name="count")
)

display(
    resolved.loc[
        decimal_mask,
        [
            "jmg_resolved_id",
            "SOURCE_FILE",
            "original_WGSe_numeric",
            "original_WGSn_numeric",
            "resolved_longitude",
            "resolved_latitude",
            "original_vs_resolved_distance_m",
        ]
    ]
)


,original_wgs_status,count
0,WHOLE_DEGREE_COARSE,39
1,ZERO_PLACEHOLDER,9
2,DECIMAL_COORDINATE,4


,jmg_resolved_id,SOURCE_FILE,original_WGSe_numeric,original_WGSn_numeric,resolved_longitude,resolved_latitude,original_vs_resolved_distance_m
48,JMG_049,kdsg_TANAH_RUNTUH_SABAH.shp,116.541813,5.999077,116.538629,5.999817,363.953024
49,JMG_050,kdsg_TANAH_RUNTUH_SABAH.shp,116.600528,6.012806,116.597109,6.012823,380.608201
50,JMG_051,kdsg_TANAH_RUNTUH_SABAH.shp,116.601917,6.014236,116.598097,6.015133,436.927923
51,JMG_052,kdsg_TANAH_RUNTUH_SABAH.shp,116.539150,6.005770,116.538533,6.005416,79.346856


## 13. Malaysia-boundary validation

Every resolved point should fall inside or on the supplied Malaysia boundary.


In [16]:
if malaysia.crs is None:
    raise ValueError(
        "Malaysia boundary CRS is missing."
    )

resolved_boundary_crs = resolved.to_crs(malaysia.crs)

if len(malaysia) == 1:
    malaysia_geom = malaysia.geometry.iloc[0]
else:
    malaysia_geom = malaysia.dissolve().geometry.iloc[0]

resolved["inside_malaysia_boundary"] = (
    resolved_boundary_crs.geometry
    .apply(malaysia_geom.covers)
    .to_numpy()
)

display(
    resolved["inside_malaysia_boundary"]
    .value_counts(dropna=False)
    .rename_axis("inside_malaysia")
    .reset_index(name="count")
)

outside = resolved.loc[
    ~resolved["inside_malaysia_boundary"],
    [
        "jmg_resolved_id",
        "SOURCE_FILE",
        "jmg_source_event_id",
        "Negeri",
        "Daerah",
        "resolved_longitude",
        "resolved_latitude",
    ]
]

if len(outside):
    print("WARNING: Resolved points outside supplied Malaysia boundary:")
    display(outside)
else:
    print("All 52 resolved JMG occurrence points are inside/on the supplied Malaysia boundary.")


,inside_malaysia,count
0,True,52


All 52 resolved JMG occurrence points are inside/on the supplied Malaysia boundary.


## 14. Coordinate range and AOI/source summary

This checks that transformed values are geographically plausible and summarises each JMG source.


In [17]:
resolved["coordinate_plausible_malaysia"] = (
    resolved["resolved_longitude"].between(99, 120)
    & resolved["resolved_latitude"].between(0, 8.5)
)

source_summary = (
    resolved.groupby(
        ["SOURCE_FILE", "source_epsg"],
        dropna=False
    )
    .agg(
        records=("jmg_resolved_id", "size"),
        min_lon=("resolved_longitude", "min"),
        max_lon=("resolved_longitude", "max"),
        min_lat=("resolved_latitude", "min"),
        max_lat=("resolved_latitude", "max"),
        inside_malaysia=("inside_malaysia_boundary", "sum"),
        complete_dates=("source_date_complete_valid", "sum"),
    )
    .reset_index()
)

display(source_summary)

source_summary.to_csv(
    OUTPUT_DIR / "jmg_resolved_source_summary.csv",
    index=False
)

if not resolved["coordinate_plausible_malaysia"].all():
    print("WARNING: Some transformed coordinates fail the broad Malaysia envelope.")
else:
    print("All transformed coordinates pass the broad Malaysia coordinate envelope.")


,SOURCE_FILE,source_epsg,records,min_lon,max_lon,min_lat,max_lat,inside_malaysia,complete_dates
0,ch_TANAH_RUNTUH_SEMENANJUNG_.shp,3375,16,101.371947,101.400939,4.409206,4.500211,16,15
1,ipoh_TANAH_RUNTUH_SEMENANJUNG_.shp,3375,1,101.066294,101.066294,4.645719,4.645719,1,0
2,kdsg_TANAH_RUNTUH_SABAH.shp,3376,4,116.538533,116.598097,5.999817,6.015133,4,1
3,pp_TANAH_RUNTUH_SEMENANJUNG_.shp,3375,3,100.277564,100.297046,5.454412,5.467109,3,3
4,uk_TANAH_RUNTUH_SEMENANJUNG_.shp,3375,28,101.724140,101.851253,3.100022,3.246230,28,28


All transformed coordinates pass the broad Malaysia coordinate envelope.


## 15. Flag repeated IDs and repeated projected/WGS84 locations

These are **review flags only**.

Do not drop any row in this notebook.


In [18]:
resolved["duplicate_source_event_id"] = (
    resolved["jmg_source_event_id"].notna()
    & resolved["jmg_source_event_id"].duplicated(keep=False)
)

resolved["duplicate_exact_projected_xy"] = (
    resolved.duplicated(
        ["SOURCE_FILE", "source_geometry_x", "source_geometry_y"],
        keep=False
    )
)

# Round only for duplicate screening, not to replace actual coordinates.
resolved["_lon_round8"] = resolved["resolved_longitude"].round(8)
resolved["_lat_round8"] = resolved["resolved_latitude"].round(8)

resolved["duplicate_resolved_wgs84"] = (
    resolved.duplicated(
        ["_lon_round8", "_lat_round8"],
        keep=False
    )
)

duplicate_review = resolved.loc[
    resolved["duplicate_source_event_id"]
    | resolved["duplicate_exact_projected_xy"]
    | resolved["duplicate_resolved_wgs84"],
    [
        "jmg_resolved_id",
        "SOURCE_FILE",
        "source_feature_index",
        "jmg_source_event_id",
        "source_geometry_x",
        "source_geometry_y",
        "resolved_longitude",
        "resolved_latitude",
        "Negeri",
        "Daerah",
        "Hari",
        "Bulan",
        "Tahun",
        "source_date_candidate",
        "duplicate_source_event_id",
        "duplicate_exact_projected_xy",
        "duplicate_resolved_wgs84",
    ]
].copy()

print("JMG records requiring duplicate review:", len(duplicate_review))
display(duplicate_review)

duplicate_review.drop(columns="geometry", errors="ignore").to_csv(
    OUTPUT_DIR / "jmg_internal_duplicate_review.csv",
    index=False
)


JMG records requiring duplicate review: 6


,jmg_resolved_id,SOURCE_FILE,source_feature_index,jmg_source_event_id,source_geometry_x,source_geometry_y,resolved_longitude,resolved_latitude,Negeri,Daerah,Hari,Bulan,Tahun,source_date_candidate,duplicate_source_event_id,duplicate_exact_projected_xy,duplicate_resolved_wgs84
36,JMG_037,ch_TANAH_RUNTUH_SEMENANJUNG_.shp,8,AY44201_LS_05,376296.00,497079.0,101.379137,4.491970,Pahang,Cameron Highlands,19,Disember,2021.0,2021-12-19,True,False,False
37,JMG_038,ch_TANAH_RUNTUH_SEMENANJUNG_.shp,9,AY44201_LS_05,376020.00,495378.0,101.376696,4.476578,Pahang,Cameron Highlands,19,Disember,2021.0,2021-12-19,True,False,False
41,JMG_042,ch_TANAH_RUNTUH_SEMENANJUNG_.shp,13,AY44201_M_01_SF_01,377057.00,497079.0,101.385995,4.491990,Pahang,Cameron Highlands,29,Mei,2021.0,2021-05-29,False,True,True
42,JMG_043,ch_TANAH_RUNTUH_SEMENANJUNG_.shp,14,AY44201_M_02_SF_01,377057.00,497079.0,101.385995,4.491990,Pahang,Cameron Highlands,6,Januari,2021.0,2021-01-06,False,True,True
44,JMG_045,pp_TANAH_RUNTUH_SEMENANJUNG_.shp,0,FH2101_M_01_SF_01,256719.82,603963.5,100.297046,5.454412,Pulau Pinang,Timor Laut,30,Oktober,2021.0,2021-10-30,True,False,False
45,JMG_046,pp_TANAH_RUNTUH_SEMENANJUNG_.shp,1,FH2101_M_01_SF_01,256544.00,604104.0,100.295453,5.455675,Pulau Pinang,Timor Laut,28,September,2021.0,2021-09-28,True,False,False


## 16. Map resolved JMG inventory

The figure is saved even if your VS Code notebook renderer cannot display PNG output inline.


In [19]:
malaysia_4326 = malaysia.to_crs("EPSG:4326")

fig, ax = plt.subplots(figsize=(12, 9))

malaysia_4326.boundary.plot(
    ax=ax,
    linewidth=0.8
)

resolved.plot(
    ax=ax,
    markersize=35,
    alpha=0.8,
    label="JMG resolved occurrences"
)

ax.set_title(
    "JMG Landslide Inventory - Resolved Source Geometry"
)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend()
ax.grid(True, alpha=0.2)

plt.tight_layout()

map_path = (
    OUTPUT_DIR
    / "jmg_coordinate_resolution_map.png"
)

fig.savefig(
    map_path,
    dpi=180,
    bbox_inches="tight"
)

plt.close(fig)

print("Map successfully saved to:")
print(map_path)
print()
print(
    "If VS Code cannot render PNG output inline, "
    "open the saved file manually."
)


Map successfully saved to:
C:\Landslide_FYP\Landslide_FYP_Development\01_preprocessing\jmg_coordinate_resolution_outputs\jmg_coordinate_resolution_map.png

If VS Code cannot render PNG output inline, open the saved file manually.


## 17. Save resolved JMG inventory

Two versions are saved:

### Audit version
Contains many helper/check fields.

### Clean version
Contains original source attributes plus resolved WGS84 fields and essential provenance/quality flags.

No duplicate rows are removed.


In [20]:
# ------------------------------------------------------------
# Full audit GeoJSON
# ------------------------------------------------------------
audit_geojson_path = (
    OUTPUT_DIR
    / "jmg_coordinate_resolution_full_audit.geojson"
)

# GeoJSON cannot safely store timezone-aware or unsupported objects.
resolved_export = resolved.copy()

for col in resolved_export.columns:
    if pd.api.types.is_datetime64_any_dtype(resolved_export[col]):
        resolved_export[col] = (
            resolved_export[col]
            .astype("string")
            .replace("<NA>", None)
        )

resolved_export.to_file(
    audit_geojson_path,
    driver="GeoJSON"
)

# ------------------------------------------------------------
# Clean spatial inventory
# ------------------------------------------------------------
helper_drop = [
    "_day",
    "_month",
    "_year",
    "_lon_round8",
    "_lat_round8",
]

clean_jmg = resolved.drop(
    columns=helper_drop,
    errors="ignore"
).copy()

clean_geojson_path = (
    CLEAN_DIR
    / "jmg_inventory_wgs84_resolved.geojson"
)

clean_csv_path = (
    CLEAN_DIR
    / "jmg_inventory_wgs84_resolved.csv"
)

clean_export = clean_jmg.copy()

# Store candidate date as text for broad GeoJSON compatibility.
if "source_date_candidate" in clean_export.columns:
    clean_export["source_date_candidate"] = (
        clean_export["source_date_candidate"]
        .astype("string")
        .replace("<NA>", None)
    )

clean_export.to_file(
    clean_geojson_path,
    driver="GeoJSON"
)

clean_jmg.drop(
    columns="geometry",
    errors="ignore"
).to_csv(
    clean_csv_path,
    index=False
)

print("Saved full audit GeoJSON:")
print(audit_geojson_path)
print()
print("Saved cleaned resolved JMG GeoJSON:")
print(clean_geojson_path)
print()
print("Saved cleaned resolved JMG CSV:")
print(clean_csv_path)


Saved full audit GeoJSON:
C:\Landslide_FYP\Landslide_FYP_Development\01_preprocessing\jmg_coordinate_resolution_outputs\jmg_coordinate_resolution_full_audit.geojson

Saved cleaned resolved JMG GeoJSON:
C:\Landslide_FYP\Landslide_FYP_Development\02_cleaned_inventory\jmg_inventory_wgs84_resolved.geojson

Saved cleaned resolved JMG CSV:
C:\Landslide_FYP\Landslide_FYP_Development\02_cleaned_inventory\jmg_inventory_wgs84_resolved.csv


## 18. Generate Stage-2 resolution report


In [21]:
resolution_summary = pd.DataFrame({
    "check": [
        "JMG CSV rows",
        "Source occurrence shapefiles",
        "Source occurrence features",
        "Resolved WGS84 records",
        "Inside Malaysia boundary",
        "Outside Malaysia boundary",
        "Plausible Malaysia coordinates",
        "Complete valid source dates",
        "Missing/incomplete source dates",
        "Records with source event ID",
        "Records missing source event ID",
        "Rows flagged repeated source ID",
        "Rows flagged repeated projected location",
        "Rows flagged repeated resolved WGS84 location",
    ],
    "value": [
        len(jmg_csv_work),
        len(source_layers),
        sum(len(g) for g in source_layers.values()),
        len(resolved),
        int(resolved["inside_malaysia_boundary"].sum()),
        int((~resolved["inside_malaysia_boundary"]).sum()),
        int(resolved["coordinate_plausible_malaysia"].sum()),
        int(resolved["source_date_complete_valid"].sum()),
        int((~resolved["source_date_complete_valid"]).sum()),
        int(resolved["jmg_source_event_id"].notna().sum()),
        int(resolved["jmg_source_event_id"].isna().sum()),
        int(resolved["duplicate_source_event_id"].sum()),
        int(resolved["duplicate_exact_projected_xy"].sum()),
        int(resolved["duplicate_resolved_wgs84"].sum()),
    ]
})

display(resolution_summary)

resolution_summary.to_csv(
    OUTPUT_DIR / "jmg_coordinate_resolution_summary.csv",
    index=False
)

print("\nSTAGE-2 READINESS DECISION")
print("--------------------------")

all_spatially_resolved = (
    len(resolved) == EXPECTED_TOTAL
    and resolved["inside_malaysia_boundary"].all()
    and resolved["coordinate_plausible_malaysia"].all()
)

if all_spatially_resolved:
    print("PASS: All 52 JMG records have source-based WGS84 geometry.")
else:
    print("REVIEW REQUIRED: One or more JMG records need spatial review.")

print()
print("Do NOT remove duplicate candidates yet.")
print(
    "Next notebook: 03_positive_inventory_deduplication.ipynb"
)
print(
    "That notebook will compare the resolved 52 JMG points against "
    "the 236 NASA COOLR points before deciding the final positive count."
)


,check,value
0,JMG CSV rows,52
1,Source occurrence shapefiles,5
2,Source occurrence features,52
3,Resolved WGS84 records,52
4,Inside Malaysia boundary,52
5,Outside Malaysia boundary,0
6,Plausible Malaysia coordinates,52
7,Complete valid source dates,47
8,Missing/incomplete source dates,5
9,Records with source event ID,49



STAGE-2 READINESS DECISION
--------------------------
PASS: All 52 JMG records have source-based WGS84 geometry.

Do NOT remove duplicate candidates yet.
Next notebook: 03_positive_inventory_deduplication.ipynb
That notebook will compare the resolved 52 JMG points against the 236 NASA COOLR points before deciding the final positive count.


# Expected Stage-2 output structure

After successful execution:

```text
Landslide_FYP_Development/
├── 01_preprocessing/
│   ├── 01_inventory_audit.ipynb
│   ├── 02_jmg_coordinate_resolution.ipynb
│   │
│   └── jmg_coordinate_resolution_outputs/
│       ├── jmg_source_crs_inventory.csv
│       ├── jmg_geometry_vs_rso_check.csv
│       ├── jmg_csv_to_source_match_audit.csv
│       ├── jmg_resolved_source_summary.csv
│       ├── jmg_internal_duplicate_review.csv
│       ├── jmg_coordinate_resolution_map.png
│       ├── jmg_coordinate_resolution_full_audit.geojson
│       └── jmg_coordinate_resolution_summary.csv
│
└── 02_cleaned_inventory/
    ├── jmg_inventory_wgs84_resolved.geojson
    └── jmg_inventory_wgs84_resolved.csv
```

## Next stage

`03_positive_inventory_deduplication.ipynb`

That stage will:

1. load 236 NASA COOLR points,
2. load the 52 resolved JMG points,
3. review within-source duplicates,
4. calculate JMG ↔ COOLR spatial proximity,
5. compare dates and descriptive attributes where available,
6. assign duplicate groups conservatively,
7. create the final reviewed positive inventory,
8. calculate the final unique-positive count,
9. map national positive-event coverage.

Pseudo-absence generation comes **after** this.
